# vLLM Support

## Summary

[vLLM](https://github.com/vllm-project/vllm) is a high-performance inference
library that uses PagedAttention, continuous batching, and a custom execution
engine to serve HuggingFace-compatible language models at much higher
throughput than a plain `transformers` run. NNsight ships with a full vLLM
backend so you can read and edit model internals **without giving up vLLM's
speed** — your intervention code is compiled, serialized onto each request, and
run inside vLLM's own worker against the real weights.

This guide covers every way to use the integration:

- Instantiating the `VLLM` wrapper (sync and async)
- Reading and writing module activations inside a trace
- Batching multiple prompts with per-prompt `tracer.invoke()` blocks
- The `logits` and `samples` eproperties for inspecting/editing sampling
- Multi-token generation with `tracer.all()` and `tracer.iter[...]`
- Bulk activation extraction with `tracer.cache()`
- Async mode (`mode="async"`) for streaming per-request saves
- Serving over HTTP (`serve=url`) from a GPU-less client
- Tensor parallelism and the Ray executor for multi-GPU / multi-node
- Limitations and gotchas

## When to use

`VLLM` is the right tool when you need high-throughput generation or want to
run the same interventions across many prompts. A few things to know before
choosing it over `LanguageModel`:

- **Text generation only.** NNsight supports vLLM's text-generation models
  (list [here](https://docs.vllm.ai/en/latest/models/supported_models/#text-generation)).
  Multimodal and image-generation models aren't supported yet.
- **No gradients.** vLLM's paged-attention path doesn't build an autograd
  graph, so `.backward()` / `.grad` aren't available. Use `LanguageModel` for
  gradient work.
- **vLLM ≠ transformers numerically.** Fused kernels, attention
  implementations, and quantization defaults make vLLM's outputs differ
  slightly from `transformers`. The interventions are correct in both — the
  baselines just aren't bit-identical.
- **One prompt per invoke.** Each `tracer.invoke(...)` is exactly one vLLM
  request; batch by putting many invokes inside one trace (vLLM's scheduler
  batches them on the GPU).

## Setup

The vLLM integration is an optional extra:

<pre><code>pip install "nnsight[vllm]"
</code></pre>

This pulls in `vllm` and `triton` alongside NNsight.

> **Running this notebook needs a CUDA GPU.** `vllm` only runs on GPU, so the
> cells below are left unexecuted here. Install `nnsight[vllm]` on a GPU machine
> to run them end to end.

In [ ]:
import torch

## Instantiating a vLLM model

`VLLM(...)` takes a HuggingFace repo id plus any `vllm.LLM` / `AsyncEngineArgs`
keyword arguments. `dispatch=True` builds the engine now; with `dispatch=False`
(the default) only a lightweight **meta** tree is built — no GPU memory is used
until the first trace.

In [ ]:
from nnsight.modeling.vllm import VLLM

vllm = VLLM(
    "openai-community/gpt2",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.15,
    dispatch=True,
)

print(vllm)

The wrapper looks and behaves like any other NNsight model: the underlying
vLLM engine is reachable via `vllm.vllm_entrypoint`, the tokenizer via
`vllm.tokenizer`, and the module envoys (`vllm.transformer.h[...]`,
`vllm.lm_head`, etc.) are what you intervene on.

Two model-level values are worth knowing about upfront — they're **eproperties**
(the same descriptor mechanism behind a module's `.output`/`.input`) and only
meaningful inside a trace:

- `vllm.logits` — the pre-sampling logits for the current step
- `vllm.samples` — the token ids the sampler drew this step

These (not `tracer.result`, which vLLM doesn't serve) are how you read
generated output. We'll use both below.

## Basic interventions

Read or write any module's inputs/outputs inside a `model.trace(...)` block,
just as with `LanguageModel`:

In [ ]:
prompt = "The Eiffel Tower is in the city of"

with vllm.trace(prompt, temperature=0.0, top_p=1) as tracer:
    # Read: a middle-layer MLP output.
    mlp_out = vllm.transformer.h[6].mlp.output.save()

    # Write: zero the last hidden state going into lm_head. vLLM runs under
    # inference_mode, so module outputs are read-only — clone, mutate, assign back.
    ln_out = vllm.transformer.ln_f.output.clone()
    ln_out[-1, :] = 0
    vllm.transformer.ln_f.output = ln_out

    # Read the first predicted token.
    next_token = vllm.logits.argmax(dim=-1).save()

print("mlp_out shape:", mlp_out.shape)
print("predicted token:", repr(vllm.tokenizer.decode(next_token)))

Three things to notice:

1. **Flat token layout.** vLLM concatenates all tokens from all requests into a
   single `[total_tokens, hidden]` tensor — there is no batch dimension. That's
   why the slice above is `[-1, :]`, not `[:, -1, :]` as with `LanguageModel`.
   Inside a single-prompt invoke NNsight narrows your request's own rows, so you
   still work in per-request coordinates.
2. **Clone before writing.** Module outputs are read-only under
   `inference_mode`. Clone, mutate the clone, and assign the whole tensor back —
   NNsight feeds the assigned value into the next layer.
3. **Save by *name*.** `.save()` (or `nnsight.save(...)`) marks a value to come
   back to your process, keyed by **its variable name** — so you must *bind* it
   (`x = ....save()`). A bare `vllm.logits.save()` with nothing on the left marks
   the value but leaves no name to return it under, so it silently never comes
   back. Calling `.save()` outside a trace raises.

## Batching multiple prompts with invoke blocks

With `LanguageModel` you can pass a list of prompts to one invoke. With vLLM you
**can't** — each `tracer.invoke(...)` is one vLLM request. To run several
prompts, open one `with tracer.invoke(...)` block per prompt inside a single
`trace()`; vLLM's continuous batcher processes them together.

The rule specific to vLLM: **each invoke's intervention is serialized into its
own request and run in a separate worker.** A container declared at trace scope
and appended inside each invoke does *not* merge back — the appends happen in
different workers. Give each invoke its **own** `.save()`d variable:

In [ ]:
with vllm.trace(temperature=0.0, top_p=1) as tracer:
    with tracer.invoke("The Eiffel Tower is in the city of"):
        eiffel = vllm.logits.argmax(dim=-1).save()
    with tracer.invoke("Madison Square Garden is in the city of"):
        msg = vllm.logits.argmax(dim=-1).save()
    with tracer.invoke("The Colosseum is in the city of"):
        colosseum = vllm.logits.argmax(dim=-1).save()

for label, token in [
    ("The Eiffel Tower is in the city of", eiffel),
    ("Madison Square Garden is in the city of", msg),
    ("The Colosseum is in the city of", colosseum),
]:
    print(f"{label!r:<46} → {vllm.tokenizer.decode(token)!r}")

Every invoke runs its own intervention code, but vLLM batches the underlying
forward passes. You can pass **different sampling params per invoke**
(`tracer.invoke(prompt, temperature=0.8)`) — per-invoke params override the
trace-level ones.

> For a **dynamic** number of prompts you can't give each invoke a distinct
> variable name in one trace, and a shared container won't merge. Instead fire
> each prompt as its own async trace concurrently (`asyncio.gather`, see
> [Async mode](#async-mode-mode-async) below) — the engine still batches the
> concurrent requests, and each one's saves arrive on its own finished output.

### Collecting multiple generated tokens per invoke

The same per-invoke rule applies when generating several tokens: give each
invoke its own saved list and append inside `tracer.all()`:

In [ ]:
with vllm.trace(temperature=0.0, top_p=1, max_tokens=3) as tracer:
    with tracer.invoke("The Eiffel Tower is in"):
        eiffel = list().save()
        for _ in tracer.all():
            eiffel.append(vllm.samples.item())
    with tracer.invoke("Madison Square Garden is in"):
        msg = list().save()
        for _ in tracer.all():
            msg.append(vllm.samples.item())
    with tracer.invoke("The Colosseum is in"):
        colosseum = list().save()
        for _ in tracer.all():
            colosseum.append(vllm.samples.item())

for label, toks in [
    ("The Eiffel Tower is in", eiffel),
    ("Madison Square Garden is in", msg),
    ("The Colosseum is in", colosseum),
]:
    print(f"{label!r:<32} → {vllm.tokenizer.decode(toks)!r}")

Key points:

- **Save per invoke, not into a shared container.** Each invoke's `list()` is
  `.save()`d inside its own block, so it comes back under its own name.
- **`tracer.all()` iterates every generation step** — loop over it to run the
  body once per decoded token.
- **Per-invoke sampling params override the trace-level ones.**

## Accessing and editing logits and sampled tokens

`vllm.logits` and `vllm.samples` sit at the end of the pipeline:

- `vllm.logits` — the pre-sampling logits for the current step (`[vocab_size]`
  per invoke)
- `vllm.samples` — the scalar token id actually sampled

In [ ]:
with vllm.trace(
    "Madison Square Garden is located in",
    temperature=0.8,
    top_p=0.95,
    max_tokens=3,
) as tracer:
    step_logits = list().save()
    step_samples = list().save()

    for _ in tracer.all():
        step_logits.append(vllm.logits)
        step_samples.append(vllm.samples.item())

for i, (l, s) in enumerate(zip(step_logits, step_samples)):
    print(f"step {i}: top-1 via argmax={l.argmax().item():5d}  sampled={s:5d}"
          f"  ({vllm.tokenizer.decode(s)!r})")

With `temperature=0.8` the sampled token often differs from `argmax(logits)`.
You can also **write** into `vllm.logits` or `vllm.samples` to force a
decision — e.g. `vllm.samples = torch.full_like(vllm.samples, token_id)` pins
the token the engine continues generation from.

## Multi-token generation with `.all()` and `.iter[...]`

`tracer.all()` runs its body on **every** generation step:

In [ ]:
with vllm.trace("Hello world", max_tokens=5) as tracer:
    sampled = list().save()
    for _ in tracer.all():
        sampled.append(vllm.samples.item())

print("tokens:", sampled)
print("decoded:", vllm.tokenizer.decode(sampled))

`tracer.iter[slice]` runs its body only on specific steps — handy for
intervening on (or observing) a window of tokens:

In [ ]:
prompt = "The Eiffel Tower is in the city of"
mlp = vllm.transformer.h[6].mlp

with vllm.trace(prompt, max_tokens=6) as tracer:
    hidden_states = list().save()

    # Zero the MLP output on steps 2-4 (inclusive-exclusive).
    for _ in tracer.iter[2:5]:
        masked = mlp.output.clone()
        masked[-1] = 0
        mlp.output = masked
        hidden_states.append(mlp.output)

print(f"captured {len(hidden_states)} steps")

Inside `tracer.all()` / `tracer.iter[...]` you can both **read** (append to a
saved list) and **write** (mutate `.output`). Writes apply only on the steps
the iterator selects.

## Bulk activation extraction with `tracer.cache()`

The patterns above append values to a saved list one at a time. For dense
per-layer capture — "give me the residual stream at layer 6 for every token" —
use `tracer.cache()`, which registers a hook on each target module and collects
activations into a `CacheView` keyed by module path.

In [ ]:
target_layer = vllm.transformer.h[6]

with vllm.trace(
    "The Eiffel Tower is in the city of",
    temperature=0.0,
    max_tokens=8,
) as tracer:
    # tracer.cache() saves the view for you — no extra .save() needed.
    cache = tracer.cache(modules=[target_layer])

print("cached module paths:", list(cache.keys()))

# The cache is keyed by Envoy path. For GPT-2 under vLLM that's
# "model.transformer.h.6" — the "model" prefix is the wrapper's root namespace.
# When a module fires multiple times (prefill + each decode step) the entry is a
# list of Entry objects; concatenate their hidden states for every captured token.
entry = cache[next(iter(cache.keys()))]
entries = entry if isinstance(entry, list) else [entry]

def hidden_states(e):
    out = e.output
    return out[0] if isinstance(out, tuple) else out

all_hs = torch.cat([hidden_states(e) for e in entries], dim=0)
print(f"captured {len(entries)} forward passes")
print(f"total hidden-state shape: {tuple(all_hs.shape)}  "
      f"(prefill tokens + one row per decode step)")

A few things to know about `tracer.cache()`:

- **Multiple modules.** Pass a list (`modules=[layer_a, layer_b]`), or omit it
  to cache *every* module.
- **Inputs too.** `tracer.cache(include_inputs=True)` captures each module's
  `(args, kwargs)` alongside its output.
- **Device / dtype.** `device` (default `torch.device("cpu")`) and `dtype`
  move/cast captured tensors as they're collected — CPU keeps them off the GPU.
- **Per-request collection.** Each request's cache is narrowed to that request's
  rows and collected independently as it finishes, so concurrent requests don't
  share or overwrite each other's captures.

## Async mode (`mode="async"`)

Pass `mode="async"` to load the model against vLLM's `AsyncLLM` engine instead
of the sync `LLM`. The trace-writing API is identical, but the **result** is an
async generator: iterate `tracer.backend` and each yielded `RequestOutput`
streams a decode step. Saved values are attached to the **finished** output
only (fetched from the worker at that point); intermediate outputs carry an
empty `saves` dict.

In [ ]:
from nnsight.modeling.vllm import VLLM

async_vllm = VLLM(
    "openai-community/gpt2",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.15,
    dispatch=True,
    mode="async",
)

async def run_one(prompt: str):
    with async_vllm.trace(prompt, temperature=0.0, max_tokens=5) as tracer:
        # Bind saves to names — they come back keyed by variable name on the
        # finished output's `.saves` dict. tracer.cache() saves itself.
        logits = async_vllm.logits.save()
        cache = tracer.cache(modules=[async_vllm.transformer.h[6]])

    # tracer.backend is an AsyncVLLMBackend you iterate as an async generator.
    # Each yielded output is a vLLM RequestOutput; saves land on the finished one.
    final = None
    async for output in tracer.backend:
        if output.finished:
            final = output
    return final

# Jupyter already runs an asyncio loop — `await` directly. In a plain script,
# wrap this in `asyncio.run(run_one(...))`.
final = await run_one("The Eiffel Tower is in")
print("finished:", final.finished)
print("decoded:", final.outputs[0].text)
print("saves:", list(final.saves.keys()))
print("logits shape:", final.saves["logits"].shape)

`await tracer.backend` is a shortcut that drains the stream and returns just
the last (finished) output when you don't need the intermediate steps:

<pre><code>last = await tracer.backend
last.saves["logits"]
</code></pre>

Async is the right choice when firing many requests concurrently — e.g.
`asyncio.gather` over a whole dataset. vLLM's scheduler forms multi-request
batches, and each request's saves are collected and returned independently as it
finishes, rather than waiting on one collection at the very end.

Sync vs. async at a glance:

|                          | Sync (`mode="sync"`, default) | Async (`mode="async"`) |
|--------------------------|------------------------------|------------------------|
| Engine class             | `vllm.LLM`                   | `vllm.v1.engine.async_llm.AsyncLLM` |
| Best for                 | Notebook experiments, small batches | High-throughput concurrent workloads |
| Trace API                | `with model.trace(...) as t:` | same — but iterate `t.backend` after |
| Prompts per trace        | many invokes                 | one prompt (several invokes raise) |
| Saves                    | land in your locals when the block exits | on the **finished** streamed output's `.saves` |

## Serving over HTTP (`serve=url`)

The `serve/` package lets a **GPU-less** client run traces on a standalone
engine over HTTP. A server holds one dispatched async `VLLM`; a client builds
only the meta tree (no GPU, never dispatched), writes a trace as usual, and
`serve=url` sends it over. This is how you keep the heavy engine on a GPU box
while notebooks/laptops drive interventions against it.

### Start a server

The `nnsight-serve` console script (installed with `nnsight[vllm]`) brings up a
FastAPI app around a dispatched async engine:

<pre><code>nnsight-serve openai-community/gpt2 --port 6677 --gpu-memory-utilization 0.15

# optionally require an API key on every request:
nnsight-serve openai-community/gpt2 --port 6677 --api-key SECRET
</code></pre>

It binds to `127.0.0.1` by default (it runs client-supplied code, so only
expose it on a trusted network); any extra vLLM engine args are forwarded. A
`GET /health` endpoint reports readiness.

### Submit traces from a GPU-less client

The client needs no GPU — it constructs the meta tree and never dispatches:

<pre><code>from nnsight.modeling.vllm import VLLM

model = VLLM("openai-community/gpt2")          # meta tree only, never dispatched

with model.trace(
    "The Eiffel Tower is in the city of",
    serve="http://127.0.0.1:6677",
    api_key="SECRET",                          # if the server requires one
):
    model.transformer.h[6].mlp.output[:] = 0   # intervene on the server
    logits = model.logits.save()

print(model.tokenizer.decode(logits.argmax(dim=-1)))
</code></pre>

The trace is serialized in the same form the remote (NDIF) path uses, run on the
server, and its saved values pushed back into your frame — so reading a
`.save()`d variable after the block works exactly as it does locally. The
server returns **saved values only** (not generated tokens), and a build or
runtime error comes back with its real exception type and traceback; only a
transport/service failure surfaces as a `ConnectionError`.

## Tensor parallelism and multi-GPU

`VLLM(..., tensor_parallel_size=N)` shards the model across `N` GPUs on one
node. NNsight handles the sharded-tensor semantics for you: intervention code
always sees the **full** gathered tensor (not a shard), and writes are
re-sharded before being handed back to vLLM. Your tracing code is identical to
the `tp=1` case.

This isn't run inline (this notebook is single-GPU), but the pattern is drop-in
— with two GPUs visible, just set `tensor_parallel_size=2`:

<pre><code>from nnsight.modeling.vllm import VLLM

vllm_tp2 = VLLM(
    "facebook/opt-6.7b",
    tensor_parallel_size=2,
    gpu_memory_utilization=0.85,
    dispatch=True,
)

with vllm_tp2.trace("Hello world", max_tokens=5) as tracer:
    mid = vllm_tp2.model.decoder.layers[16].output.save()
</code></pre>

Under the hood, NNsight's `VLLMBatcher` brackets every parallel linear
(`ColumnParallelLinear`, `RowParallelLinear`) with pre/post hooks. Before your
code reads a sharded value it all-gathers (or all-reduces) the shards into the
full tensor — the same on every rank — runs your intervention against that whole
tensor, and on the way out re-shards so vLLM's forward resumes with the shape it
expects. Every GPU runs the same intervention on the same complete tensor.

### Multi-node with Ray

For tensor parallelism across multiple machines, use vLLM's **stock** Ray
executor — it's a plain vLLM engine kwarg, forwarded through:

<pre><code>vllm_multinode = VLLM(
    "meta-llama/Llama-3.1-70B",
    tensor_parallel_size=8,
    distributed_executor_backend="ray",
    dispatch=True,
)
</code></pre>

NNsight adds no custom Ray executor — it rides vLLM's own Ray path, so cluster
setup (starting `ray`, `RAY_ADDRESS`, worker placement) is exactly as documented
by vLLM for [multi-node serving](https://docs.vllm.ai/en/latest/serving/distributed_serving.html).
Your tracing code doesn't change; the mediator payload rides Ray's transport
alongside the request. NNsight's collection carries saved values home over Ray's
RPC just as it does over multiprocessing.

## Limitations and gotchas

- **No gradients.** vLLM's paged-attention kernels don't retain a graph, so
  `.backward()`, `.grad`, and gradient-based ops aren't supported. Use
  `LanguageModel`.
- **vLLM ≠ transformers numerically.** Fused kernels, attention
  implementations, and quantization defaults mean you shouldn't expect
  exact-match outputs between `VLLM` and `LanguageModel` for the same input. The
  interventions are correct in both — the baselines just differ.
- **One prompt per invoke.** `tracer.invoke(prompt)` takes one string / token-id
  list; batch by looping invokes inside a trace. A list in one invoke raises.
- **`tracer.result` is not served on vLLM.** Read generated output via
  `vllm.logits` / `vllm.samples` (or the streamed `RequestOutput` in async) —
  reading `tracer.result` would park a worker forever.
- **Save by name; a bare `.save()` returns nothing.** Bind every save
  (`x = ....save()`); an unbound `vllm.logits.save()` silently doesn't come back.
- **An empty `tracer.invoke()` with interventions raises** (its work would
  vanish); a do-nothing empty invoke is a harmless no-op.
- **A typo'd sampling kwarg raises** (`trace(temperatur=0.0)` → `TypeError`)
  rather than being silently ignored.
- **`enforce_eager` is forced.** CUDA graphs replay a fixed kernel sequence and
  skip Python, so hooks can't fire inside one — the engine runs eager. This
  costs some decode throughput.
- **Async requires `asyncio`.** `mode="async"` traces are iterated inside a
  coroutine — `await` in a notebook (Jupyter runs a loop) or `asyncio.run(...)`
  in a script.

## See also

**Other feature guides:**

- [Tracer fundamentals](1_getting.ipynb) — how traces, invokes, and `.save()`
  work in general
- [Multi-token generation](4_multiple_token.ipynb) — deeper coverage of
  `tracer.all()` / `tracer.iter[...]`
- [Cache](16_cache.ipynb) — `tracer.cache()` in depth

**End-to-end examples:**

- [`ndif-team/nnsight-vllm-demos`](https://github.com/ndif-team/nnsight-vllm-demos) —
  runnable demos of the vLLM integration, including a streaming chat UI and an
  SAE-steering example that edits activations mid-generation behind a real chat
  server.